# 02 - MOFA: do the factors relate to AML survival, independent of age?

Run from the `notebooks/` folder, in the `laml` environment, after `scripts/02` to `04` have been run.

Plan: (1) pick the most variable features, (2) train MOFA, (3) see how much variance each factor explains in each layer,
(4) ask what each factor *is* biologically (clinical variables, top genes and probes), (5) test each factor against survival,
with age in the model, (6) check that the result survives a few robustness checks.

MOFA never sees survival. Survival is only compared with the factors afterwards, so the factors cannot be tuned to the outcome.

In [ ]:
import os
import anndata as ad
import mofax
import mudata as md
import muon as mu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines import CoxPHFitter, KaplanMeierFitter

plt.rcParams["figure.dpi"] = 110

## 0. Settings

In [ ]:
RULE = "floor"        # survival-time-0 rule used in scripts/02 ("floor" = 158 patients)
DATA = "../data/processed/"
OUT = DATA + "mofa/"
TABLES = "../results/tables/"
FIG = "../results/figures/"
for folder in (OUT, TABLES, FIG):
    os.makedirs(folder, exist_ok=True)

N_MRNA = 5000         # most variable genes kept
N_METH = 5000         # most variable methylation probes kept (same number: neither layer dominates)
MIN_MEAN_MRNA = 1.0   # genes with mean log2(RSEM+1) below this are barely expressed
N_FACTORS = 15        # starting number of factors; unneeded ones shrink to ~0%
MIN_R2 = 2.0          # a factor is "kept" if it explains >= this % of one layer
SEED = 42
RETRAIN = True        # False = reuse the saved model if it exists (saves time when re-running the notebook)

## 1. Load the cleaned data (same patients, same order in every table)

In [ ]:
mrna = pd.read_pickle(DATA + f"mrna_{RULE}.pkl")                          # genes  x patients
meth = pd.read_pickle(DATA + f"methylation_filtered_{RULE}.pkl")          # probes x patients (sex-chr probes already removed)
surv = pd.read_csv(DATA + f"survival_{RULE}.csv", index_col="PatientID")  # Survival (days), Death (1 = died)
clin = pd.read_csv(DATA + f"clinical_{RULE}.csv", index_col="PatientID")  # age, gender, cyto_risk, fab, ...

patients = list(mrna.columns)
assert patients == list(meth.columns), "mRNA and methylation patients differ"
assert set(patients) == set(surv.index) == set(clin.index), "survival/clinical patients differ"
surv = surv.loc[patients]
clin = clin.loc[patients]

print(f"{len(patients)} patients | {mrna.shape[0]} genes | {meth.shape[0]} probes | {int(surv['Death'].sum())} deaths")

## 2. Keep the most variable features

Features that barely change between patients cannot reveal patient-to-patient patterns, and 380,000 probes would drown 20,000 genes.
Methylation is converted from beta (0-1) to M-values, `log2(beta / (1 - beta))`: beta is squeezed against 0 and 1, M-values are
closer to bell-shaped, which is what MOFA's Gaussian model assumes. Variance is taken on the same scale that MOFA will see.

In [ ]:
# mRNA: drop barely-expressed genes, then keep the N_MRNA most variable
mrna = mrna[mrna.mean(axis=1) >= MIN_MEAN_MRNA]
mrna = mrna.loc[mrna.var(axis=1).nlargest(N_MRNA).index]

# methylation: beta -> M-value, then keep the N_METH most variable probes
beta = np.clip(meth.to_numpy(dtype="float64"), 0.001, 0.999)
mval = pd.DataFrame(np.log2(beta / (1 - beta)), index=meth.index, columns=patients)
mval = mval.loc[mval.var(axis=1).nlargest(N_METH).index]

print(f"kept {len(mrna)} genes and {len(mval)} probes")

## 3. Train MOFA

`scale_views=True` gives each layer the same total variance, so the layer with more extreme numbers does not win by size alone.
`convergence_mode="slow"` trains longer, as in the paper. One seed here; earlier runs with two seeds gave the same result.

In [ ]:
def make_view(table):
    # AnnData wants rows = patients, columns = features
    return ad.AnnData(X=table.T.to_numpy(dtype="float32"),
                      obs=pd.DataFrame(index=patients),
                      var=pd.DataFrame(index=table.index))

model_file = OUT + "mofa_model.hdf5"
if RETRAIN or not os.path.exists(model_file):
    mdata = md.MuData({"mrna": make_view(mrna), "methylation": make_view(mval)})
    mu.tl.mofa(
        mdata,
        use_obs="union",
        n_factors=N_FACTORS,
        likelihoods=["gaussian", "gaussian"],   # one per layer, order: mrna, methylation
        scale_views=True,
        convergence_mode="slow",
        seed=SEED,
        outfile=model_file,
        save_data=False,
    )

## 4. How much variance does each factor explain in each layer?

In [ ]:
model = mofax.mofa_model(model_file)
r2 = model.get_r2()                                    # % variance: Factor, View, R2
r2_table = r2.pivot_table(index="Factor", columns="View", values="R2")
r2_table = r2_table.loc[sorted(r2_table.index, key=lambda f: int(f.replace("Factor", "")))]

print("Total % variance explained per layer:")
print(r2_table.sum().round(1).to_string())
r2_table.round(2)

In [ ]:
keep = [f for f in r2_table.index if r2_table.loc[f].max() >= MIN_R2]
print(f"{len(keep)} factors explain >= {MIN_R2}% of at least one layer: {keep}")

# bar chart: one bar per factor, one colour per layer
r2_table.loc[keep].plot(kind="bar", figsize=(8, 3.5), color=["#4c72b0", "#dd8452"])
plt.ylabel("% variance explained"); plt.xlabel("")
plt.title("Variance explained per factor and layer")
plt.tight_layout(); plt.savefig(FIG + "mofa_variance_explained.png", dpi=150); plt.show()

r2_table.loc[keep].round(2).to_csv(TABLES + "mofa_variance_explained.csv")

**How to read it:** a factor with variance in both layers is a pattern the two layers share (for example, a leukemia subtype that
changes both expression and methylation). A factor in one layer only is specific to that layer. Factors below the cutoff are
noise-level and are not tested, which also limits the number of tests below.

## 5. What is each factor? Compare with clinical variables

A factor that tracks age, sex or FAB subtype may be a known biology (or a confounder) rather than a new finding.
Tests: Spearman correlation for age and white cell count; Mann-Whitney for sex; Kruskal-Wallis for cytogenetic risk and FAB subtype.
Each p-value is then corrected for the number of tests (Benjamini-Hochberg).

In [ ]:
factors = model.get_factors(df=True).loc[patients]                     # patients x factors
Z = (factors - factors.mean()) / factors.std()                         # 1 unit = 1 SD, so factors are comparable

def bh(p):
    # Benjamini-Hochberg adjusted p-values (plain numpy)
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adj = p[order] * len(p) / np.arange(1, len(p) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty_like(adj); out[order] = np.minimum(adj, 1)
    return out

def kruskal_p(values, groups):
    d = pd.DataFrame({"v": values, "g": groups}).dropna()
    parts = [x["v"].to_numpy() for _, x in d.groupby("g") if len(x) >= 3]   # skip tiny groups
    return stats.kruskal(*parts).pvalue if len(parts) > 1 else np.nan

rows = []
for f in keep:
    v = Z[f]
    sex = clin["gender"]
    levels = sex.dropna().unique()
    rows.append({
        "factor": f,
        "age_rho": stats.spearmanr(v, clin["age"], nan_policy="omit")[0],
        "age_p": stats.spearmanr(v, clin["age"], nan_policy="omit")[1],
        "wbc_p": stats.spearmanr(v, clin["wbc"], nan_policy="omit")[1],
        "sex_p": stats.mannwhitneyu(v[sex == levels[0]], v[sex == levels[1]]).pvalue,
        "cyto_risk_p": kruskal_p(v, clin["cyto_risk"]),
        "fab_p": kruskal_p(v, clin["fab"]),
    })
assoc = pd.DataFrame(rows).set_index("factor")

# correct all p-values together
pcols = ["age_p", "wbc_p", "sex_p", "cyto_risk_p", "fab_p"]
adj = pd.DataFrame(bh(assoc[pcols].to_numpy().ravel()).reshape(len(assoc), len(pcols)),
                   index=assoc.index, columns=[c.replace("_p", "_q") for c in pcols])
assoc = assoc.join(adj)
assoc.to_csv(TABLES + "mofa_factors_vs_clinical.csv")
assoc[["age_rho"] + list(adj.columns)].round(4)

A small `q` (below 0.05) means the factor tracks that clinical variable. FAB groups with fewer than 3 patients are skipped.

## 6. Are the factors related to survival, independent of age?

Cox proportional-hazards model, one factor at a time: first alone, then with age added (age matters a lot in AML; older patients
do worse and also differ in biology). Factors are in SD units, so a hazard ratio (HR) is "per 1 SD higher factor value".
Testing one factor at a time keeps the model small: with about 100 deaths, putting all 15 factors in one model leaves too few
events per variable. p-values are corrected across the factors tested (Benjamini-Hochberg).

The sign of a factor is arbitrary in MOFA (a factor and its mirror image are the same pattern), so only the size and the p-value
carry meaning, not whether the HR is above or below 1.

In [ ]:
def cox_table(mask=None):
    # age-adjusted and unadjusted Cox model for each kept factor, on the patients selected by mask
    use = np.ones(len(patients), dtype=bool) if mask is None else np.asarray(mask)
    base = pd.concat([surv, clin[["age"]]], axis=1).loc[use]
    base["age10"] = base["age"] / 10                       # hazard ratio per 10 years of age
    rows = []
    for f in keep:
        d = base.assign(factor=Z.loc[use, f]).dropna(subset=["age10", "factor"])
        res = {"factor": f}
        for name, cols in (("alone", ["factor"]), ("age_adj", ["factor", "age10"])):
            cph = CoxPHFitter().fit(d[cols + ["Survival", "Death"]], duration_col="Survival", event_col="Death")
            s = cph.summary.loc["factor"]
            res[f"HR_{name}"] = s["exp(coef)"]
            res[f"ci_low_{name}"] = s["exp(coef) lower 95%"]
            res[f"ci_high_{name}"] = s["exp(coef) upper 95%"]
            res[f"p_{name}"] = s["p"]
        rows.append(res)
    out = pd.DataFrame(rows).set_index("factor")
    out["q_alone"] = bh(out["p_alone"])
    out["q_age_adj"] = bh(out["p_age_adj"])
    return out

cox = cox_table()
cox.to_csv(TABLES + "mofa_cox_per_factor.csv")
cox[["HR_alone", "p_alone", "q_alone", "HR_age_adj", "ci_low_age_adj", "ci_high_age_adj", "p_age_adj", "q_age_adj"]].round(4)

In [ ]:
# Reference only: age on its own, so you can judge the factors against a known predictor
d = pd.concat([surv, clin[["age"]]], axis=1).dropna()
d["age10"] = d["age"] / 10
age_only = CoxPHFitter().fit(d[["age10", "Survival", "Death"]], duration_col="Survival", event_col="Death")
age_only.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(4)

**How to read it:** if `q_age_adj` is small, the factor carries survival information that age does not explain.
If `q_alone` is small but `q_age_adj` is not, the factor was mostly standing in for age.
This is a within-sample association (the same patients trained MOFA and are tested), not proof of prediction on new patients.

## 7. Display: Kaplan-Meier curves for the strongest factors

The curves split patients at the median of the factor. The split is for display only; the p-value that counts is the Cox one above.
A median split is data-independent, unlike a cutoff picked from the survival times.

In [ ]:
top = cox.sort_values("p_age_adj").index[:3]
fig, axes = plt.subplots(1, len(top), figsize=(4.2 * len(top), 3.5), sharey=True)
axes = np.atleast_1d(axes)
for ax, f in zip(axes, top):
    high = Z[f] > Z[f].median()
    for flag, label in ((True, "high"), (False, "low")):
        KaplanMeierFitter().fit(surv.loc[high == flag, "Survival"], surv.loc[high == flag, "Death"],
                                label=f"{f} {label}").plot_survival_function(ax=ax, ci_show=False)
    ax.set_title(f"{f}  (age-adj. p = {cox.loc[f, 'p_age_adj']:.3g})", fontsize=10)
    ax.set_xlabel("days")
axes[0].set_ylabel("overall survival")
plt.tight_layout(); plt.savefig(FIG + "mofa_km_top_factors.png", dpi=150); plt.show()

## 8. What drives the important factors? Top genes and probes

Weights show how strongly each feature contributes to a factor (sign = direction). Look at the genes first; use the probe
annotation (`probeMap`) to map CpG sites to genes. Run this for the factors that matter in sections 5-6.

In [ ]:
def top_features(factor, view, n=10):
    w = model.get_weights(views=view, factors=factor, df=True)[factor]
    return pd.concat([w.nlargest(n), w.nsmallest(n)]).sort_values()

def plot_top(factor, view, n=10):
    t = top_features(factor, view, n)
    plt.figure(figsize=(5, 5))
    plt.barh(t.index, t.values, color=["tomato" if v > 0 else "steelblue" for v in t.values])
    plt.axvline(0, color="black", lw=0.8)
    plt.xlabel(f"{factor} weight"); plt.title(f"Top features: {factor} ({view})")
    plt.tight_layout(); plt.show()

FACTOR_TO_READ = top[0]                       # change to any factor name, e.g. "Factor3"
plot_top(FACTOR_TO_READ, "mrna")
plot_top(FACTOR_TO_READ, "methylation")

# save the top 25 of each sign, per view, for the factors in the survival table
out = []
for f in keep:
    for view in ("mrna", "methylation"):
        w = model.get_weights(views=view, factors=f, df=True)[f]
        for feat, val in pd.concat([w.nlargest(25), w.nsmallest(25)]).items():
            out.append({"factor": f, "view": view, "feature": feat, "weight": val})
pd.DataFrame(out).to_csv(TABLES + "mofa_top_features.csv", index=False)

## 9. Robustness: does the result survive removing doubtful patients?

Repeat the age-adjusted Cox models (a) without patients whose survival time was floored from 0 to 1 day, and (b) without FAB M3
patients (M3 is a distinct leukemia type with its own treatment and outcome, as far as I know - check the FAB code against the
clinical documentation). A factor that only works with these patients included is fragile.

In [ ]:
zero_time = (surv["Survival"] <= 1).to_numpy()
is_m3 = (clin["fab"] == "M3").to_numpy()
print(f"floored (time = 1 day): {zero_time.sum()} patients | FAB M3: {is_m3.sum()} patients")
if is_m3.sum() == 0:
    print("WARNING: no 'M3' found; check the values in clin['fab'] before trusting section (b)")

check = pd.DataFrame({
    "p_all": cox["p_age_adj"],
    "p_no_zero_time": cox_table(~zero_time)["p_age_adj"],
    "p_no_M3": cox_table(~is_m3)["p_age_adj"],
})
check.round(4)

## 10. Notes

- Everything is one random start (seed 42). Earlier on the full data two seeds gave the same ELBO and factor count.
- Factors, weights and the saved tables go to `data/processed/mofa/` (git-ignored) and `results/tables/` (committed).
- Not done: SNP / cross-reactive probe filtering, external validation, prediction on held-out patients.